# 🌿 Plant Disease Classifier — Comparación de Resultados

**Equipo:** Antonio · Alondra · Paulo · Arlette  
**Dataset:** [PlantVillage (Kaggle)](https://www.kaggle.com/datasets/arjuntejaswi/plant-village)  
**Objetivo de negocio:** Maximizar **Recall** — minimizar falsos negativos (plantas enfermas no detectadas).

---

## Distribución de Experimentos

| Modelo | Estrategia | Responsable |
|---|---|---|
| ResNet50 | Straightforward | Alondra |
| ResNet50 | Fine-tuning | Paulo |
| ResNet50 | Warm-up | Antonio |
| DenseNet121 | Straightforward | Paulo |
| DenseNet121 | Fine-tuning | Arlette |
| DenseNet121 | Warm-up | Antonio |
| VGG16 | Straightforward | Alondra |
| VGG16 | Fine-tuning | Alondra |
| VGG16 | Warm-up | Arlette |

In [ ]:
import sys
from pathlib import Path

# Asegurar que src/ sea importable desde notebooks/
sys.path.insert(0, str(Path().resolve().parent))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

EXPERIMENTS_DIR = Path('../experiments')
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / 'figures').mkdir(exist_ok=True)
(REPORTS_DIR / 'results').mkdir(exist_ok=True)

print('Directorios configurados correctamente.')

## 1. Cargar todos los resultados

In [ ]:
def load_all_metrics(experiments_dir: Path) -> pd.DataFrame:
    """Carga todos los metrics.json del árbol de experimentos."""
    records = []
    for metrics_file in sorted(experiments_dir.rglob('metrics.json')):
        with open(metrics_file) as f:
            data = json.load(f)
        records.append(data)

    if not records:
        print(f'⚠️  No se encontraron resultados en {experiments_dir}')
        print('   Ejecuta primero: python train.py --model <model> --strategy <strategy> --member <member>')
        return pd.DataFrame()

    df = pd.DataFrame(records)
    cols_order = ['experiment', 'member', 'model', 'strategy',
                  'recall', 'precision', 'f1_score', 'accuracy', 'n_test']
    existing = [c for c in cols_order if c in df.columns]
    return df[existing].sort_values('recall', ascending=False).reset_index(drop=True)


df = load_all_metrics(EXPERIMENTS_DIR)

if not df.empty:
    print(f'Total de experimentos cargados: {len(df)}')
    display(df)

## 2. Tabla resumen — ordenada por Recall (métrica prioritaria)

In [ ]:
if not df.empty:
    # Colorear la columna recall (verde = alto, rojo = bajo)
    def highlight_recall(val):
        color = f'background-color: rgba(76, 175, 80, {min(val, 1.0):.2f})' if val >= 0.85 else \
                f'background-color: rgba(244, 67, 54, {1 - val:.2f})'
        return color

    styled = df.style\
        .applymap(highlight_recall, subset=['recall'])\
        .set_caption('Resultados ordenados por Recall ← Prioridad de Negocio')\
        .format({'recall': '{:.4f}', 'precision': '{:.4f}',
                 'f1_score': '{:.4f}', 'accuracy': '{:.4f}'})

    display(styled)

## 3. Mejor resultado por modelo

In [ ]:
if not df.empty and 'model' in df.columns:
    print(f"{'Modelo':<15} {'Estrategia':<20} {'Responsable':<12} {'Recall':>8} {'F1':>8} {'Precision':>10} {'Accuracy':>10}")
    print('-' * 85)

    for model in ['resnet50', 'densenet121', 'vgg16']:
        subset = df[df['model'] == model]
        if not subset.empty:
            best = subset.iloc[0]  # ya ordenado por recall
            print(
                f"  {model:<13} {best.get('strategy', ''):<20} {best.get('member', ''):<12}"
                f" {best['recall']:>8.4f} {best['f1_score']:>8.4f}"
                f" {best['precision']:>10.4f} {best['accuracy']:>10.4f}"
            )

    # Ganador global
    print()
    winner = df.iloc[0]
    print(f"🏆 MEJOR EXPERIMENTO: {winner['experiment']}")
    print(f"   Recall    : {winner['recall']:.4f}")
    print(f"   F1-Score  : {winner['f1_score']:.4f}")
    print(f"   Responsable: {winner.get('member', 'N/A')}")

## 4. Gráficas de comparación

In [ ]:
if not df.empty:
    metrics = ['recall', 'precision', 'f1_score', 'accuracy']
    labels_map = {
        'recall': 'Recall ← PRIORIDAD',
        'precision': 'Precision',
        'f1_score': 'F1-Score',
        'accuracy': 'Accuracy'
    }
    colors_map = {
        'recall': '#4CAF50',
        'precision': '#2196F3',
        'f1_score': '#FF9800',
        'accuracy': '#9C27B0'
    }

    fig, axes = plt.subplots(2, 2, figsize=(18, 13))
    fig.suptitle('Comparación de Todos los Experimentos', fontsize=15, fontweight='bold', y=1.01)

    for ax, metric in zip(axes.flatten(), metrics):
        df_sorted = df.sort_values(metric, ascending=True)
        bars = ax.barh(
            df_sorted['experiment'],
            df_sorted[metric],
            color=colors_map[metric],
            alpha=0.85,
            edgecolor='white',
            linewidth=0.5
        )
        # Anotar valores
        for bar, val in zip(bars, df_sorted[metric]):
            ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                    f'{val:.3f}', va='center', ha='left', fontsize=8)

        ax.set_title(labels_map[metric], fontsize=12, fontweight='bold')
        ax.set_xlim(0, 1.05)
        ax.axvline(x=0.90, color='red', linestyle='--', alpha=0.6, linewidth=1.2,
                   label='Target ≥ 0.90')
        ax.legend(fontsize=9)
        ax.grid(axis='x', alpha=0.3)
        ax.set_xlabel(metric.capitalize())

    plt.tight_layout()
    fig_path = REPORTS_DIR / 'figures' / 'model_comparison.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura guardada: {fig_path}')

## 5. Comparación por modelo y estrategia

In [ ]:
if not df.empty and 'model' in df.columns and 'strategy' in df.columns:
    fig, ax = plt.subplots(figsize=(14, 6))

    x = np.arange(len(df))
    width = 0.2

    for i, (metric, color) in enumerate([
        ('recall', '#4CAF50'),
        ('precision', '#2196F3'),
        ('f1_score', '#FF9800'),
        ('accuracy', '#9C27B0'),
    ]):
        offset = (i - 1.5) * width
        bars = ax.bar(x + offset, df[metric], width, label=metric.replace('_', ' ').title(),
                      color=color, alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(df['experiment'], rotation=45, ha='right', fontsize=9)
    ax.axhline(y=0.90, color='red', linestyle='--', alpha=0.5, linewidth=1, label='Target: 0.90')
    ax.set_ylim(0, 1.05)
    ax.set_title('Métricas por Experimento', fontsize=13, fontweight='bold')
    ax.set_ylabel('Valor')
    ax.legend(loc='lower right')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    fig_path = REPORTS_DIR / 'figures' / 'grouped_metrics.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura guardada: {fig_path}')

## 6. Recall por estrategia — ¿cuál funciona mejor?

In [ ]:
if not df.empty and 'strategy' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Por estrategia
    strategy_recall = df.groupby('strategy')['recall'].agg(['mean', 'max', 'min']).round(4)
    strategy_recall.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#8BC34A', '#CDDC39'],
                         alpha=0.85, edgecolor='white')
    axes[0].set_title('Recall por Estrategia de Entrenamiento', fontweight='bold')
    axes[0].set_ylabel('Recall')
    axes[0].set_ylim(0, 1.05)
    axes[0].axhline(0.90, color='red', linestyle='--', alpha=0.5)
    axes[0].legend(['Media', 'Máximo', 'Mínimo'])
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20, ha='right')
    axes[0].grid(axis='y', alpha=0.3)

    # Por modelo
    if 'model' in df.columns:
        model_recall = df.groupby('model')['recall'].agg(['mean', 'max', 'min']).round(4)
        model_recall.plot(kind='bar', ax=axes[1], color=['#2196F3', '#03A9F4', '#00BCD4'],
                          alpha=0.85, edgecolor='white')
        axes[1].set_title('Recall por Modelo (Backbone)', fontweight='bold')
        axes[1].set_ylabel('Recall')
        axes[1].set_ylim(0, 1.05)
        axes[1].axhline(0.90, color='red', linestyle='--', alpha=0.5)
        axes[1].legend(['Media', 'Máximo', 'Mínimo'])
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20, ha='right')
        axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    fig_path = REPORTS_DIR / 'figures' / 'recall_by_strategy_and_model.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()

    print('\n📊 Recall promedio por estrategia:')
    display(strategy_recall)

## 7. Curvas de entrenamiento (history)

In [ ]:
def plot_history_overlay(experiments_dir: Path, metric: str = 'val_recall'):
    """Superpone las curvas de val_recall de todos los experimentos."""
    fig, ax = plt.subplots(figsize=(14, 6))

    cmap = plt.get_cmap('tab10')
    found = 0

    for i, history_file in enumerate(sorted(experiments_dir.rglob('history.json'))):
        with open(history_file) as f:
            history = json.load(f)

        if metric not in history:
            continue

        vals = history[metric]
        epochs = range(1, len(vals) + 1)
        label = history_file.parent.relative_to(experiments_dir)
        ax.plot(epochs, vals, label=str(label), color=cmap(i % 10), linewidth=2)
        found += 1

    if found == 0:
        print(f'No se encontraron histories con la métrica "{metric}".')
        plt.close()
        return

    ax.axhline(y=0.90, color='red', linestyle='--', alpha=0.5, label='Target ≥ 0.90')
    ax.set_title(f'Evolución de {metric} durante entrenamiento', fontsize=13, fontweight='bold')
    ax.set_xlabel('Época')
    ax.set_ylabel(metric.replace('val_', 'Val ').title())
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()

    fig_path = REPORTS_DIR / 'figures' / f'{metric}_overlay.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura guardada: {fig_path}')


plot_history_overlay(EXPERIMENTS_DIR, metric='val_recall')

## 8. Matrices de confusión

In [ ]:
confusion_matrices = list(EXPERIMENTS_DIR.rglob('confusion_matrix.npy'))

if not confusion_matrices:
    print('⚠️  No se encontraron matrices de confusión (.npy).')
    print('   Asegúrate de que los experimentos hayan completado la evaluación.')
else:
    n = len(confusion_matrices)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).flatten() if n > 1 else [axes]

    CLASS_NAMES = ['healthy', 'diseased']

    for ax, cm_path in zip(axes, confusion_matrices):
        cm = np.load(cm_path)
        # Título = experiments/<member>/<experiment>
        parts = cm_path.parts
        title = '/'.join(parts[-3:-1]) if len(parts) >= 3 else str(cm_path)

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    ax=ax, linewidths=0.5, annot_kws={'size': 12})
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_ylabel('Real')
        fn = cm[1, 0] if cm.shape == (2, 2) else 0
        ax.set_xlabel(f'Predicho\n⚠️ FN={fn}')

    # Ocultar ejes sin datos
    for ax in axes[n:]:
        ax.set_visible(False)

    plt.suptitle('Matrices de Confusión — Todos los Experimentos', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig_path = REPORTS_DIR / 'figures' / 'all_confusion_matrices.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura guardada: {fig_path}')

## 9. Exportar tabla de resultados

In [ ]:
if not df.empty:
    csv_path = REPORTS_DIR / 'results' / 'final_comparison.csv'
    df.to_csv(csv_path, index=True)
    print(f'✅ Resultados exportados a: {csv_path}')

    # Tabla final en consola
    print('\n' + '=' * 80)
    print('  RANKING FINAL — Ordenado por Recall')
    print('=' * 80)
    print(df.to_string(index=True))
    print()

    # Conclusión automática
    best = df.iloc[0]
    print(f'\n🏆 Mejor experimento: {best["experiment"]}')
    print(f'   Recall    : {best["recall"]:.4f}')
    print(f'   F1-Score  : {best["f1_score"]:.4f}')
    print(f'   Precision : {best["precision"]:.4f}')
    print(f'   Accuracy  : {best["accuracy"]:.4f}')
    print(f'   Responsable: {best.get("member", "N/A")}')